# Unified Nepali Pipeline: Phase 2 (Pre-training) ➔ Phase 3 (Reasoning)
This notebook uses the ALREADY TRAINED Nepali tokenizer from Phase 1, but fixes the catastrophic Phase 2 wrapper bug where the Unicode Character-level merges were being ignored. By dynamically switching to Unicode BPE inside `TokenizerWrapper`, this notebook successfully re-tokenizes the corpus, re-trains Phase 2 correctly, and evaluates Phase 3.

### Instructions to run on Kaggle:
1. Turn on **GPU (P100 or T4x2)** in the Kaggle Accelerator settings.
2. Ensure **Internet** is toggled ON.
3. Click "Run All".

In [ ]:
!pip install -q datasets torch numpy tqdm sacrebleu rouge-score seaborn
!git clone -b Phase-3 https://github.com/CL3-410/individual-project-ManyaJain111.git
%cd individual-project-ManyaJain111

# Download the pre-trained tokenizers (which we will use as-is)
!python phase3/download_models.py

## 1. Phase 2: Tokenize Corpus & Pre-train Model

In [ ]:
import os, json, numpy as np
from datasets import load_dataset
from tqdm.auto import tqdm
from phase2.src.tokenizer_wrapper import TokenizerWrapper

print("Downloading Nepali corpus...")
ds = load_dataset("manyajain111/nepali-lma-corpus")

# The wrapper now automatically detects the `custom_unicode_bpe_v1` format!
tokenizer = TokenizerWrapper("checkpoints/pretrained/nepali/tokenizer.json")
print(f"Tokenizer loaded! Vocab size: {tokenizer.vocab_size}")

def tokenize_to_bin(split_name, out_path):
    print(f"Tokenizing {split_name}...")
    all_ids = []
    for item in tqdm(ds[split_name]):
        ids = tokenizer.encode(item['text'])
        all_ids.extend(ids)
        # Append unknown ID or a separator as newline was dropped
        all_ids.append(tokenizer.tokenizer.unk_id)
    arr = np.array(all_ids, dtype=np.uint16)
    arr.tofile(out_path)
    print(f"Saved {out_path} ({len(arr):,} tokens)")

os.makedirs("/kaggle/working", exist_ok=True)
tokenize_to_bin("train", "/kaggle/working/nepali_train.bin")
tokenize_to_bin("validation", "/kaggle/working/nepali_val.bin")

In [ ]:
# Update config to match exact vocab size from the loaded tokenizer
with open("phase2/config/model_l_config.json", "r") as f:
    cfg = json.load(f)
cfg["vocab_size"] = tokenizer.vocab_size
with open("phase2/config/model_l_config.json", "w") as f:
    json.dump(cfg, f, indent=4)

# Train! (Adjust max_steps as needed for your compute budget)
!python phase2/src/train.py \
    --config phase2/config/model_l_config.json \
    --tokenizer checkpoints/pretrained/nepali/tokenizer.json \
    --dataset manyajain111/nepali-lma-corpus \
    --output_dir /kaggle/working/checkpoints_p2 \
    --batch_size 16 \
    --grad_accum 8 \
    --max_steps 61000

## 2. Phase 3: Reasoning Finetuning & Exact Match Evaluation

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from phase2.src.model import GPTLanguageModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load the newly pretrained model
with open("phase2/config/model_l_config.json", "r") as f:
    config = json.load(f)

model = GPTLanguageModel(**config)

# Load the final checkpoint from Phase 2 (Update 'model_final.pt' to your saved checkpoint name)
ckpt = torch.load("/kaggle/working/checkpoints_p2/model_final.pt", map_location=device)
model.load_state_dict(ckpt.get('model_state_dict', ckpt))
model.to(device)
print("Phase 2 Model Loaded for Finetuning!")

In [ ]:
class ReasoningDataset(Dataset):
    def __init__(self, jsonl_path, tokenizer, max_len=256):
        self.data = []
        with open(jsonl_path, 'r', encoding='utf-8') as f:
            for line in f:
                d = json.loads(line)
                prompt, answer = d['prompt'], d['answer']
                full_text = f"{prompt}\nउत्तर: {answer}"
                tokens = tokenizer.encode(full_text)

                prompt_text = f"{prompt}\nउत्तर: "
                prompt_len = len(tokenizer.encode(prompt_text))

                if len(tokens) > max_len:
                    tokens = tokens[:max_len]

                input_ids = tokens[:-1]
                labels = tokens[1:]
                mask_len = max(prompt_len - 1, 0)
                for i in range(min(mask_len, len(labels))):
                    labels[i] = -100

                pad_len = (max_len - 1) - len(input_ids)
                if pad_len > 0:
                    input_ids = input_ids + [0] * pad_len
                    labels = labels + [-100] * pad_len

                self.data.append((torch.tensor(input_ids, dtype=torch.long),
                                   torch.tensor(labels, dtype=torch.long)))

    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]

train_ds = ReasoningDataset("phase3/data/ne/train.jsonl", tokenizer)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_ds = ReasoningDataset("phase3/data/ne/val.jsonl", tokenizer)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

print("Starting Finetuning...")
for epoch in range(1, 4):
    model.train()
    train_loss = 0
    for x, y in tqdm(train_loader, desc=f"Epoch {epoch} Train"):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits, loss = model(x, targets=y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for x, y in tqdm(val_loader, desc=f"Epoch {epoch} Val"):
            x, y = x.to(device), y.to(device)
            _, loss = model(x, targets=y)
            val_loss += loss.item()
            
    print(f"Epoch {epoch} | Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss/len(val_loader):.4f}")

In [ ]:
def evaluate_exact_match_nepali(model, tokenizer, jsonl_path, num_samples=200):
    model.eval()
    correct_by_tier = {1: 0, 2: 0, 3: 0}
    total_by_tier = {1: 0, 2: 0, 3: 0}
    
    with open(jsonl_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()[:num_samples]
        
    with torch.no_grad():
        for line in tqdm(lines, desc="Evaluating Exact Match"):
            d = json.loads(line)
            prompt = d['prompt']
            label = d['label'] 
            tier = d.get('tier', 1)
            
            context_str = f"{prompt}\nउत्तर: "
            idx = torch.tensor([tokenizer.encode(context_str)], dtype=torch.long).to(device)
            
            # Subword generation
            out_idx = model.greedy_generate(idx, max_new_tokens=40)
            generated_str = tokenizer.decode(out_idx[0].tolist())
            
            # Slicing bug officially fixed:
            decoded_context = tokenizer.decode(tokenizer.encode(context_str))
            gen_answer = generated_str[len(decoded_context):]
            
            if label in gen_answer:
                correct_by_tier[tier] += 1
            total_by_tier[tier] += 1
            
    print(f"\n--- NEPALI RESULTS ---")
    total_correct = sum(correct_by_tier.values())
    total = sum(total_by_tier.values())
    print(f"Overall Accuracy: {total_correct/total*100:.2f}%")
    for t in [1, 2, 3]:
        if total_by_tier[t] > 0:
            print(f"  Tier {t}: {correct_by_tier[t]/total_by_tier[t]*100:.2f}%")

evaluate_exact_match_nepali(model, tokenizer, "phase3/data/ne/test.jsonl", num_samples=500)